# Puyo Puyo AI — Self-Play 強化学習（Google Colab GPU）

このノートブックは Google Colab の GPU（T4 / A100）を使って
`puyo-trainer` の `self-play` バイナリを実行し、
学習済みモデルを Google Drive に保存・GitHub にプッシュします。

## 前提条件

1. **GPU ランタイムを選択してください**  
   メニュー → `ランタイム` → `ランタイムのタイプを変更` → `T4 GPU`（または A100）

2. **Colab シークレットに `GITHUB_TOKEN` を設定**  
   左サイドバー 🔑 → `GITHUB_TOKEN` を追加  
   GitHub の Fine-grained PAT、権限: `Contents: Read and write`

## セッション切れ後の再開手順

Colab のセッションは約 90 分（無料）でリセットされます。  
再接続後は以下のセルを順に実行してください：

| セル | 毎回必要か | 補足 |
|------|-----------|------|
| 2: Drive マウント | ✅ 必要 | |
| 3: GPU 確認 | ✅ 推奨 | スキップ可 |
| 4: Rust インストール | ✅ 必要 | セッション揮発 |
| 5: リポジトリ | ✅ 必要 | 既存なら `git pull` のみ |
| 6: Drive から復元 | ✅ 必要 | 前回のモデルを引き継ぐ |
| 7: ビルド | ✅ 必要 | ビルドキャッシュも揮発 |
| 8: self-play 実行 | ✅ 必要 | |
| 9: Drive に保存 | ✅ 必要 | セッション切れ対策 |
| 10: GitHub にプッシュ | ✅ 推奨 | main ブランチに自動コミット |


In [ ]:
# ============================================================
# セル 2: Google Drive マウント
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# Drive 上の保存ディレクトリ（変更可能）
DRIVE_DIR = '/content/drive/MyDrive/puyopuyo_ai'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive 保存先: {DRIVE_DIR}')

In [ ]:
# ============================================================
# セル 3: GPU / CUDA 環境確認
# ============================================================
import subprocess
import glob
import os

print('=== GPU 確認 ===')
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('GPU が見つかりません。ランタイムを GPU に変更してください。')
    raise SystemExit('GPU ランタイムが必要です')

print('\n=== CUDA ライブラリ確認 ===')
cuda_libs = (
    glob.glob('/usr/local/cuda*/lib64/libcuda.so*') +
    glob.glob('/usr/lib/x86_64-linux-gnu/libcuda.so*')
)
print('libcuda:', cuda_libs if cuda_libs else '見つかりません')

# burn/cuda-jit が参照する CUDA_PATH を設定
cuda_dirs = sorted(glob.glob('/usr/local/cuda*'))
CUDA_PATH = cuda_dirs[-1] if cuda_dirs else '/usr/local/cuda'

os.environ['CUDA_PATH'] = CUDA_PATH
os.environ['CUDA_HOME'] = CUDA_PATH

print(f'\nCUDA_PATH={CUDA_PATH}')
print('nvcc（コンパイラ）は burn/cuda-jit の JIT 実行には不要です')

In [ ]:
# ============================================================
# セル 4: Rust インストール
# ============================================================
import subprocess
import os
import glob

# rustup でインストール
result = subprocess.run(
    'curl --proto =https --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable',
    shell=True, capture_output=False
)

# PATH を永続化
cargo_bin = '/root/.cargo/bin'
os.environ['PATH'] = f"{cargo_bin}:{os.environ['PATH']}"

# CUDA_PATH も再設定（セルをスキップして実行された場合に備えて）
cuda_dirs = sorted(glob.glob('/usr/local/cuda*'))
CUDA_PATH = cuda_dirs[-1] if cuda_dirs else '/usr/local/cuda'
os.environ['CUDA_PATH'] = CUDA_PATH
os.environ['CUDA_HOME'] = CUDA_PATH

# バージョン確認
for cmd in [['rustc', '--version'], ['cargo', '--version']]:
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout.strip() if r.returncode == 0 else f'{cmd[0]} が見つかりません')

In [ ]:
# ============================================================
# セル 5: リポジトリ clone / pull
# Colab 左サイドバー 🔑 に GITHUB_TOKEN（Fine-grained PAT）を設定してください
# 必要な権限: Contents (read/write)
# ============================================================
import subprocess
import os
from google.colab import userdata

REPO_DIR = '/content/puyopuyo-ai'

# PAT でクローン（push 時にも認証に使用）
token = userdata.get('GITHUB_TOKEN')
REPO_URL = f'https://{token}@github.com/hfappmaker/puyopuyo-ai.git'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('リポジトリ既存 → git pull')
    result = subprocess.run(['git', 'pull'], cwd=REPO_DIR, text=True)
else:
    print('リポジトリ clone 中...')
    result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], text=True)
    if result.returncode != 0:
        raise RuntimeError('clone 失敗。GITHUB_TOKEN の権限を確認してください。')
    print('clone 完了')

# artifacts ディレクトリを確認
artifacts_dir = os.path.join(REPO_DIR, 'artifacts')
os.makedirs(artifacts_dir, exist_ok=True)
print('\nartifacts/ の内容:')
for f in sorted(os.listdir(artifacts_dir)):
    size = os.path.getsize(os.path.join(artifacts_dir, f))
    print(f'  {f}: {size:,} bytes')

In [ ]:
# ============================================================
# セル 6: Drive からモデルを artifacts/ に復元
# （初回実行時はスキップ可。2回目以降は実行して前回のモデルを引き継ぐ）
# ============================================================
import shutil
import os

REPO_DIR = '/content/puyopuyo-ai'
DRIVE_DIR = '/content/drive/MyDrive/puyopuyo_ai'

files_to_restore = [
    'puyo_model.bin',
    'puyo_model_selfplay.bin',
    'norm_params.txt',
]

print('=== Drive からアーティファクト復元 ===')
for fname in files_to_restore:
    src = os.path.join(DRIVE_DIR, fname)
    dst = os.path.join(REPO_DIR, 'artifacts', fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size = os.path.getsize(dst)
        print(f'  復元: {fname} ({size:,} bytes)')
    else:
        print(f'  スキップ（Drive になし）: {fname}')

# 必須ファイルの確認
print('\n=== 必須ファイル確認 ===')
for fname in ['puyo_model.bin', 'norm_params.txt']:
    path = os.path.join(REPO_DIR, 'artifacts', fname)
    if os.path.exists(path):
        print(f'  OK: {fname}')
    else:
        print(f'  WARNING: {fname} が artifacts/ にありません')
        print('  → git clone で取得されているはず（リポジトリに含まれている場合）')
        print('  → なければ先に generate-data → train を実行してください')

In [ ]:
# ============================================================
# セル 7: ビルド（GPU）
# 初回は 10〜20 分かかります。Colab の出力に進捗が表示されます。
# ============================================================
import subprocess
import os
import glob

REPO_DIR = '/content/puyopuyo-ai'

# 環境変数を再確認
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ.get('PATH', '')}"
cuda_dirs = sorted(glob.glob('/usr/local/cuda*'))
CUDA_PATH = cuda_dirs[-1] if cuda_dirs else '/usr/local/cuda'
os.environ['CUDA_PATH'] = CUDA_PATH
os.environ['CUDA_HOME'] = CUDA_PATH

print(f'CUDA_PATH={CUDA_PATH}')
print('GPU ビルド開始（初回は時間がかかります）...')

result = subprocess.run(
    ['cargo', 'build', '--release', '-p', 'puyo-trainer', '--bin', 'self-play'],
    cwd=REPO_DIR,
    env=os.environ,
)

if result.returncode == 0:
    binary = os.path.join(REPO_DIR, 'target/release/self-play')
    size = os.path.getsize(binary)
    print(f'\nビルド成功: {binary} ({size:,} bytes)')
else:
    print(f'\nGPU ビルド失敗 (returncode={result.returncode})')
    print('→ 下のフォールバックセルで CPU ビルドを試みてください')

In [ ]:
# ============================================================
# セル 7b: CPU フォールバックビルド（GPU ビルドが失敗した場合のみ実行）
# ============================================================
import subprocess
import os

REPO_DIR = '/content/puyopuyo-ai'
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ.get('PATH', '')}"

print('CPU ビルド開始（GPU が使えない環境向け）...')
result = subprocess.run(
    [
        'cargo', 'build', '--release',
        '-p', 'puyo-trainer', '--bin', 'self-play',
        '--no-default-features', '--features', 'cpu',
    ],
    cwd=REPO_DIR,
    env=os.environ,
)
if result.returncode == 0:
    binary = os.path.join(REPO_DIR, 'target/release/self-play')
    print(f'CPU ビルド成功: {binary}')
else:
    print('CPU ビルドも失敗しました。エラーログを確認してください。')

In [ ]:
# ============================================================
# セル 8: self-play 実行
# リアルタイムでログが流れます。5000 ゲーム完了まで待ちます。
# ============================================================
import subprocess
import os
import time
import glob

REPO_DIR = '/content/puyopuyo-ai'
binary = os.path.join(REPO_DIR, 'target/release/self-play')

if not os.path.exists(binary):
    raise FileNotFoundError('バイナリが見つかりません。セル 7 のビルドを先に実行してください。')

# CUDA_PATH を再設定（JIT コンパイル時に参照される）
cuda_dirs = sorted(glob.glob('/usr/local/cuda*'))
CUDA_PATH = cuda_dirs[-1] if cuda_dirs else '/usr/local/cuda'
os.environ['CUDA_PATH'] = CUDA_PATH
os.environ['CUDA_HOME'] = CUDA_PATH

print('self-play 開始')
print('設定: NUM_GAMES=5000, LEARNING_RATE=1e-4, BATCH_SIZE=512')
print('注意: CUDA JIT の初回コンパイル（最初の数ゲーム）はやや遅いです')
print('-' * 60)

start_time = time.time()
output_lines = []

# cwd をリポジトリルートに設定（artifacts/ への相対パス参照のため必須）
process = subprocess.Popen(
    [binary],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ,
)

for line in process.stdout:
    print(line, end='', flush=True)
    output_lines.append(line)

process.wait()
elapsed = time.time() - start_time

print('-' * 60)
if process.returncode == 0:
    print(f'self-play 完了！経過時間: {elapsed / 60:.1f} 分')
else:
    print(f'self-play 失敗 (returncode={process.returncode})')
    print('最後の 15 行:')
    for line in output_lines[-15:]:
        print(line, end='')

In [ ]:
# ============================================================
# セル 9: 学習済みモデルを Drive に保存
# self-play 完了後すぐに実行してください（セッション切れに備えて）
# ============================================================
import shutil
import os
import time

REPO_DIR = '/content/puyopuyo-ai'
DRIVE_DIR = '/content/drive/MyDrive/puyopuyo_ai'
os.makedirs(DRIVE_DIR, exist_ok=True)

timestamp = time.strftime('%Y%m%d_%H%M%S')

files_to_save = [
    'puyo_model_selfplay.bin',
    'puyo_model.bin',
    'norm_params.txt',
]

print('=== Drive にモデルを保存 ===')
for fname in files_to_save:
    src = os.path.join(REPO_DIR, 'artifacts', fname)
    dst = os.path.join(DRIVE_DIR, fname)

    if not os.path.exists(src):
        print(f'  スキップ（ファイルなし）: {fname}')
        continue

    # 既存ファイルをバックアップ
    if os.path.exists(dst):
        bak = os.path.join(DRIVE_DIR, f'backup_{timestamp}_{fname}')
        shutil.copy2(dst, bak)
        print(f'  バックアップ: backup_{timestamp}_{fname}')

    shutil.copy2(src, dst)
    size = os.path.getsize(dst)
    print(f'  保存: {fname} ({size:,} bytes)')

print('\n=== Drive の保存内容 ===')
for f in sorted(os.listdir(DRIVE_DIR)):
    path = os.path.join(DRIVE_DIR, f)
    size = os.path.getsize(path)
    mtime = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(os.path.getmtime(path)))
    print(f'  {f}: {size:,} bytes  ({mtime})')

## 完了！

### 保存されたファイル（Google Drive: `MyDrive/puyopuyo_ai/`）

| ファイル | 内容 |
|---------|------|
| `puyo_model_selfplay.bin` | self-play で更新されたモデル |
| `puyo_model.bin` | 元の教師あり学習モデル（バックアップ） |
| `norm_params.txt` | 正規化パラメータ |
| `backup_YYYYMMDD_HHMMSS_*.bin` | 上書き前の自動バックアップ |

### GitHub へのプッシュ（セル 10）

セル 10 を実行するには Colab シークレットに `GITHUB_TOKEN` が必要です：

1. Colab 左サイドバー 🔑 **シークレット** を開く
2. `GITHUB_TOKEN` を追加（GitHub の Fine-grained PAT）
3. PAT に必要な権限: `Contents: Read and write`

### 次のステップ（ローカルで作業する場合）

1. `git pull` でモデルを取得
2. `artifacts/puyo_model_selfplay.bin` → `artifacts/puyo_model.bin` にコピー
3. WASM を再ビルド:
   ```bash
   bash scripts/build-wasm.sh
   ```

### 追加学習（継続実行）

セル 6 で Drive から前回のモデルを復元してからセル 8 を再実行すると学習が継続されます。


In [ ]:
# ============================================================
# セル 10: 学習済みモデルを GitHub にプッシュ
# セル 9（Drive 保存）の後に実行してください
# ============================================================
import subprocess
import os
from google.colab import userdata

REPO_DIR = '/content/puyopuyo-ai'

# git の user 設定（コミットに必要）
subprocess.run(['git', 'config', 'user.email', 'colab-training@example.com'], cwd=REPO_DIR)
subprocess.run(['git', 'config', 'user.name', 'Colab Training'], cwd=REPO_DIR)

# PAT を remote URL に設定（push 認証）
token = userdata.get('GITHUB_TOKEN')
remote_url = f'https://{token}@github.com/hfappmaker/puyopuyo-ai.git'
subprocess.run(['git', 'remote', 'set-url', 'origin', remote_url], cwd=REPO_DIR)

# ステータス確認
print('=== git status ===')
result = subprocess.run(['git', 'status', '--short'], cwd=REPO_DIR, capture_output=True, text=True)
print(result.stdout or '変更なし')

# artifacts/ のモデルファイルをステージング
files_to_commit = [
    'artifacts/puyo_model_selfplay.bin',
    'artifacts/puyo_model.bin',
    'artifacts/norm_params.txt',
]
staged = []
for f in files_to_commit:
    path = os.path.join(REPO_DIR, f)
    if os.path.exists(path):
        subprocess.run(['git', 'add', f], cwd=REPO_DIR)
        staged.append(f)
        print(f'  staged: {f}')
    else:
        print(f'  スキップ（ファイルなし）: {f}')

if not staged:
    print('コミットするファイルがありません')
else:
    import time
    timestamp = time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())

    # コミット
    commit_msg = f'chore: update self-play model [{timestamp}]'
    result = subprocess.run(
        ['git', 'commit', '-m', commit_msg],
        cwd=REPO_DIR, capture_output=True, text=True
    )
    print(result.stdout.strip())
    if result.returncode != 0:
        print('コミット失敗:', result.stderr.strip())
    else:
        # プッシュ
        print('\nプッシュ中...')
        result = subprocess.run(
            ['git', 'push', 'origin', 'main'],
            cwd=REPO_DIR, capture_output=True, text=True
        )
        if result.returncode == 0:
            print('プッシュ完了！')
            print(result.stderr.strip())
        else:
            print('プッシュ失敗:')
            print(result.stderr.strip())
            print('→ GITHUB_TOKEN の Contents write 権限を確認してください')